# ift3710 Quick Playground

This notebook shows a compact end-to-end pipeline run for the `ift3710` project using a small California CEED subset.
It demonstrates preprocessing, spatial tensor visualization, and a lightweight model pass using the repository code.


## 1. Environment Setup

Verify the Python environment and required packages before running the demo.


In [ ]:
import sys
import os
import importlib.util
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'README.md').exists(), 'Please run this notebook from the repository root.'

print('Repository root:', ROOT)
print('Python version:', sys.version.replace('', ' '))

required_pkgs = [
    'torch',
    'numpy',
    'pandas',
    'matplotlib',
    'geopandas',
    'rasterio',
    'scipy',
    'datasets',
    'huggingface_hub',
    'tqdm',
    'sklearn',
]
missing = []
for pkg in required_pkgs:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

print('Missing packages:', missing)
if missing:
    print('If packages are missing, install them with the repository requirements or `uv sync`.')


## 2. Inspect Repository and Data Paths

Confirm the key preprocessing and model files are available, and locate the raw and processed data directories.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
core_files = [
    ROOT / 'src' / 'ssm1_pipeline.py',
    ROOT / 'src' / 'main_mutimodal.py',
    ROOT / 'src' / 'main.py',
    ROOT / 'src' / 'data-processing' / 'california' / 'run_pre_processing.sh',
    ROOT / 'src' / 'data-processing' / 'california' / 'ceed_maps_pipeline.py',
    ROOT / 'src' / 'data-processing' / 'california' / 'preprocess_full_pipeline.py',
]
data_dirs = [
    ROOT / 'data' / 'CEED',
    ROOT / 'data' / 'processed',
]

for path in core_files + data_dirs:
    print(path, '->', 'FOUND' if path.exists() else 'MISSING')


## 3. Create a Small CEED Data Subset

This notebook keeps the demo lightweight by processing a small year range instead of the full 1987-2020 dataset.


In [ ]:
import os
from pathlib import Path
import sys

ROOT = Path.cwd()
CALIF_DIR = ROOT / 'src' / 'data-processing' / 'california'
sys.path.insert(0, str(CALIF_DIR))

import ceed_maps_pipeline
import preprocess_full_pipeline

print('California pipeline folder:', CALIF_DIR)
print('CEED events file:', preprocess_full_pipeline.EVENTS_CSV)
print('Patch mapping file:', preprocess_full_pipeline.PATCH_CSV)

subset_year_start = 2001
subset_year_end = 2010
subset_target_year = 2010
print('Using a small subset of years:', subset_year_start, 'to', subset_year_end)

preprocess_full_pipeline.ensure_patch_csv()

print('Ready to build the California subset pipeline.')


## 4. Run California Preprocessing

Generate the 5-channel tensors and patch arrays for the small year range.


In [ ]:
import os
from pathlib import Path
import sys

ROOT = Path.cwd()
CALIF_DIR = ROOT / 'src' / 'data-processing' / 'california'
os.chdir(CALIF_DIR)

import ceed_maps_pipeline
import preprocess_full_pipeline

print('Working directory for preprocessing:', Path.cwd())

ceed_maps_pipeline.run_map_pipeline(subset_year_start, subset_year_end)

print('Loading and adapting CEED events for the subset...')
df_small = preprocess_full_pipeline.prepare_dataframe(
    preprocess_full_pipeline.load_and_adapt(subset_year_start, subset_year_end)
)

processed_dir = CALIF_DIR / 'data' / 'CEED' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
features_file = processed_dir / 'ceed_mini_train_output.pickle'
labels_file = processed_dir / 'ceed_mini_train_labels.pickle'

print('Building mini features pickle:', features_file)
preprocess_full_pipeline.build_pickle(
    df_small,
    str(features_file),
    norm_start=subset_year_start,
    target_years=[subset_target_year],
)
print('Building mini labels pickle:', labels_file)
preprocess_full_pipeline.build_labels(
    df_small,
    str(labels_file),
    target_years=[subset_target_year],
)

os.chdir(ROOT)
print('Returned to root:', Path.cwd())


## 5. Load and Visualize Spatial Tensors and Patches

Inspect one of the generated yearly tensors and a few patch examples.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
CALIF_DIR = ROOT / 'src' / 'data-processing' / 'california'
year = subset_target_year
tensor_path = CALIF_DIR / 'data' / 'processed' / 'cal_maps' / f'tensor_{year}.npy'
patches_path = CALIF_DIR / 'data' / 'processed' / 'cal_maps' / f'patches_{year}.npy'

print('Tensor file exists:', tensor_path.exists())
print('Patches file exists:', patches_path.exists())

tensor = np.load(tensor_path)
patches = np.load(patches_path)

print('Tensor shape:', tensor.shape)
print('Patches shape:', patches.shape)

channel_names = [
    'Sedimentary lithology',
    'Volcanic lithology',
    'Crystalline lithology',
    'Fault distance / edge map',
    'Earthquake magnitude density',
]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for idx, ax in enumerate(axes):
    im = ax.imshow(tensor[idx], cmap='viridis')
    ax.set_title(channel_names[idx])
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

sample_patch = patches[0]
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for idx, ax in enumerate(axes):
    im = ax.imshow(sample_patch[idx], cmap='viridis')
    ax.set_title(f'Patch channel {idx}')
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle(f'Example patch 0 for year {year}')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## 6. Run the SSM Pipeline on Processed Data

Use the repository's `src/ssm1_pipeline.py` logic with the mini preprocessed feature files.


In [ ]:
import shutil
import importlib.util
from pathlib import Path
import sys

ROOT = Path.cwd()
processed_dir = ROOT / 'src' / 'data-processing' / 'california' / 'data' / 'CEED' / 'processed'

for split in ['training', 'testing']:
    shutil.copy(
        processed_dir / 'ceed_mini_train_output.pickle',
        processed_dir / f'ceed_{split}_output.pickle',
    )
    shutil.copy(
        processed_dir / 'ceed_mini_train_labels.pickle',
        processed_dir / f'ceed_{split}_labels.pickle',
    )

ssm1_path = ROOT / 'src' / 'ssm1_pipeline.py'
spec = importlib.util.spec_from_file_location('ssm1_pipeline', ssm1_path)
ssm1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ssm1)

print('Running ssm1 pipeline with skip_preprocessing=True')
ssm1.run_pipeline(skip_preprocessing=True)
print('SSM pipeline completed. Output saved.')


## 7. Run a Lightweight Model Inference

Perform a minimal forward pass with the SafeNet embeddings model on the mini dataset.


In [ ]:
import torch
from pathlib import Path
import sys
from torch.utils.data import DataLoader

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from utils.dataset import MultimodalSafeNetDataset
from models.safenet_embeddings import SafeNetEmbeddings

processed_dir = ROOT / 'src' / 'data-processing' / 'california' / 'data' / 'CEED' / 'processed'
features_path = processed_dir / 'ceed_training_output.pickle'
labels_path = processed_dir / 'ceed_training_labels.pickle'

dataset = MultimodalSafeNetDataset(str(features_path), str(labels_path))
loader = DataLoader(dataset, batch_size=1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SafeNetEmbeddings(num_classes=4).to(device)
model.eval()

for batch_idx, (inputs, label) in enumerate(loader):
    inputs = {k: v.to(device) for k, v in inputs.items()}
    label = label.to(device)
    with torch.no_grad():
        logits = model(inputs)
    print('Batch', batch_idx, 'logits shape:', logits.shape)
    print('Label shape:', label.shape)
    break


## 8. Display Sample Evaluation Metrics

Compute simple prediction metrics on the mini dataset using the model output.


In [ ]:
import sys
from pathlib import Path
from torch.utils.data import DataLoader
import torch

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from utils.dataset import MultimodalSafeNetDataset
from models.safenet_embeddings import SafeNetEmbeddings
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

processed_dir = ROOT / 'src' / 'data-processing' / 'california' / 'data' / 'CEED' / 'processed'
dataset = MultimodalSafeNetDataset(
    str(processed_dir / 'ceed_training_output.pickle'),
    str(processed_dir / 'ceed_training_labels.pickle'),
)
loader = DataLoader(dataset, batch_size=1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SafeNetEmbeddings(num_classes=4).to(device)
model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():
    for inputs, label in loader:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits = model(inputs)
        pred = logits.argmax(dim=-1).cpu().numpy().ravel()
        true = label.cpu().numpy().ravel()
        pred_labels.extend(pred.tolist())
        true_labels.extend(true.tolist())

print('Samples:', len(true_labels))
print('Accuracy:', accuracy_score(true_labels, pred_labels))
print('Macro F1:', f1_score(true_labels, pred_labels, average='macro', zero_division=0))
print('Macro precision:', precision_score(true_labels, pred_labels, average='macro', zero_division=0))
print('Macro recall:', recall_score(true_labels, pred_labels, average='macro', zero_division=0))
